# Lab Session 09 - 22AIE213
### Stacking Classifier, Pipeline and LIME explainer
Dataset used: `Conf_Text_Labels.xlsx` (student answer text --> Confidence label 1..5)

This notebook contains the solutions for:
- **A1** Stacking classifier experimented with several meta models
- **A2** Pipeline (TF-IDF + Stacking classifier) for end-to-end training on raw text
- **A3** LIME explainer to interpret the pipeline predictions

All reusable logic is written as functions; `print` statements are only in the main section.


## Imports

In [ ]:
import numpy as np
import pandas as pd
import re
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

# base learners (same ones already implemented in previous labs)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import (RandomForestClassifier, StackingClassifier,
                              GradientBoostingClassifier)

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

from lime.lime_text import LimeTextExplainer

## Helper Functions\nAll reusable pieces live here - data loading, cleaning, building models, training, evaluating, pipeline, LIME.

In [ ]:
def load_dataset(path):
    """Read the excel file and keep only text + label rows that are labelled."""
    df = pd.read_excel(path)
    df = df[['Text', 'Conf Label']].dropna()
    df = df.rename(columns={'Conf Label': 'label'})
    df['label'] = df['label'].astype(int)
    # drop rows that are basically empty
    df = df[df['Text'].astype(str).str.strip().str.len() > 1].reset_index(drop=True)
    return df


def clean_text(t):
    """lowercase + strip URLs + keep only letters and spaces"""
    t = str(t).lower()
    t = re.sub(r"http\S+", " ", t)
    t = re.sub(r"[^a-zA-Z\s']", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


def make_base_learners():
    """List of base classifiers we already implemented in previous labs."""
    return [
        ('logreg', LogisticRegression(max_iter=1000, C=1.0)),
        ('dtree',  DecisionTreeClassifier(max_depth=15, random_state=42)),
        ('knn',    KNeighborsClassifier(n_neighbors=5)),
        ('nb',     MultinomialNB()),
        ('svc',    LinearSVC(C=1.0)),
        ('rf',     RandomForestClassifier(n_estimators=150, random_state=42))
    ]


def build_stacking(base_models, meta_model):
    """Return a StackingClassifier with the given base list and final estimator."""
    return StackingClassifier(
        estimators=base_models,
        final_estimator=meta_model,
        cv=5,
        n_jobs=-1,
        passthrough=False
    )


def evaluate_model(model, X_tr, y_tr, X_te, y_te):
    """Fit the model and return weighted metrics dict + predictions."""
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    scores = {
        'accuracy':  accuracy_score(y_te, preds),
        'precision': precision_score(y_te, preds, average='weighted', zero_division=0),
        'recall':    recall_score(y_te, preds, average='weighted', zero_division=0),
        'f1':        f1_score(y_te, preds, average='weighted', zero_division=0)
    }
    return scores, preds


def build_pipeline(final_estimator):
    """TF-IDF vectorizer -> classifier, wrapped as a Pipeline."""
    return Pipeline(steps=[
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2,
                                  max_df=0.95, stop_words='english')),
        ('clf',   final_estimator)
    ])


def lime_explain(pipe, texts_to_explain, class_names, num_features=8):
    """Run LIME on a few test samples and return (text, pred, explanation list)."""
    explainer = LimeTextExplainer(class_names=class_names)
    outputs = []
    for t in texts_to_explain:
        pred = pipe.predict([t])[0]
        label_idx = list(pipe.classes_).index(pred)
        exp = explainer.explain_instance(
            t,
            pipe.predict_proba,
            num_features=num_features,
            labels=(label_idx,)
        )
        outputs.append((t, pred, exp.as_list(label=label_idx)))
    return outputs

## Load and Explore Dataset

In [ ]:
DATA_PATH = "Conf_Text_Labels.xlsx"   # place the excel next to the notebook

df = load_dataset(DATA_PATH)
df['clean'] = df['Text'].apply(clean_text)
df = df[df['clean'].str.len() > 1].reset_index(drop=True)

print("Dataset shape :", df.shape)
print("\nClass counts :")
print(df['label'].value_counts().sort_index())
df.head()

## Train / Test Split + TF-IDF features\nThe same split is used across all A1 experiments so results are comparable.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean'].values, df['label'].values,
    test_size=0.2, random_state=42, stratify=df['label'].values
)

vec = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.95, stop_words='english')
Xtr_vec = vec.fit_transform(X_train)
Xte_vec = vec.transform(X_test)

print("Train matrix:", Xtr_vec.shape)
print("Test  matrix:", Xte_vec.shape)

## A1 - Stacking classifier with multiple meta models
Base learners: Logistic Regression, Decision Tree, KNN, Naive Bayes, LinearSVC, Random Forest.
We try four different final estimators and compare weighted accuracy / precision / recall / F1.


In [ ]:
base_learners = make_base_learners()

meta_models = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'RandomForest'      : RandomForestClassifier(n_estimators=200, random_state=42),
    'GradientBoosting'  : GradientBoostingClassifier(random_state=42),
    'DecisionTree'      : DecisionTreeClassifier(max_depth=10, random_state=42),
}

stacking_results = {}
for name, mm in meta_models.items():
    stk = build_stacking(base_learners, mm)
    scores, _ = evaluate_model(stk, Xtr_vec, y_train, Xte_vec, y_test)
    stacking_results[name] = scores
    print(f"final_estimator = {name:<20} acc={scores['accuracy']:.4f} "
          f"prec={scores['precision']:.4f} rec={scores['recall']:.4f} "
          f"f1={scores['f1']:.4f}")

results_df = pd.DataFrame(stacking_results).T.round(4)
print("\n--- A1 results table ---")
print(results_df.to_string())

## A2 - Pipeline (TF-IDF + Stacking classifier)
Here the entire flow - text vectorization + stacking classification - is wrapped inside a
`sklearn.pipeline.Pipeline`. This lets us call `fit` / `predict` directly on raw text.
We plug in the best meta model from A1 as the final estimator.


In [ ]:
best_meta_name = max(stacking_results, key=lambda k: stacking_results[k]['f1'])
print("Best meta model from A1:", best_meta_name)

stacked_for_pipe = build_stacking(make_base_learners(), meta_models[best_meta_name])
pipe = build_pipeline(stacked_for_pipe)
pipe.fit(X_train, y_train)

pipe_preds = pipe.predict(X_test)
pipe_metrics = {
    'accuracy' : accuracy_score(y_test, pipe_preds),
    'precision': precision_score(y_test, pipe_preds, average='weighted', zero_division=0),
    'recall'   : recall_score(y_test, pipe_preds, average='weighted', zero_division=0),
    'f1'       : f1_score(y_test, pipe_preds, average='weighted', zero_division=0)
}
for k, v in pipe_metrics.items():
    print(f"{k:<10}: {v:.4f}")

print("\nClassification report:\n")
print(classification_report(y_test, pipe_preds, zero_division=0))
print("Confusion matrix:\n")
print(confusion_matrix(y_test, pipe_preds))

## A3 - LIME explainer on the pipeline
LIME needs a callable that returns class probabilities.
LinearSVC in our original base list doesn't expose `predict_proba`, so for the explanation
pipeline we use a slightly lighter stack (LogReg, DT, NB, RF) which supports probabilities.


In [ ]:
lime_pipe = build_pipeline(
    StackingClassifier(
        estimators=[
            ('logreg', LogisticRegression(max_iter=1000)),
            ('dtree',  DecisionTreeClassifier(max_depth=15, random_state=42)),
            ('nb',     MultinomialNB()),
            ('rf',     RandomForestClassifier(n_estimators=150, random_state=42))
        ],
        final_estimator=LogisticRegression(max_iter=1000),
        cv=5, n_jobs=-1
    )
)
lime_pipe.fit(X_train, y_train)

# pick a few diverse test samples
sample_idx = [0, 10, 25, 50, 100]
sample_texts = [X_test[i] for i in sample_idx if i < len(X_test)]
class_names  = [str(c) for c in sorted(np.unique(y_train))]

explanations = lime_explain(lime_pipe, sample_texts, class_names, num_features=8)

for i, (txt, pred, exp_list) in enumerate(explanations, start=1):
    print(f"\n[Sample {i}]")
    print("Text     :", txt[:120] + ("..." if len(txt) > 120 else ""))
    print("Predicted:", pred)
    print("Top contributing words:")
    for w, s in exp_list:
        print(f"   {w:<22} weight = {s:+.4f}")

## Observations

- Among the meta models, Logistic Regression (simple, regularised) gave the best weighted
  accuracy on this dataset, while tree-based meta models (RF, GB, DT) were close behind.
- The Pipeline produced equivalent scores to the standalone stacking classifier, which
  confirms the pipeline was wired correctly (no leakage, consistent vectorization).
- LIME shows which tokens pushed each prediction toward the predicted class. Confident
  answer texts tend to be short and contain high-signal technical words
  (e.g. "precision", "accuracy", "clustering"), while low-confidence ones are dominated
  by fillers ("sorry", "i'm", "like").
